# Flash Attention: Tiling and Online Softmax

This notebook derives the algorithm before any JAX or Pallas code.

## Why tiling works

Attention output for query row $i$:
$$O_i = \sum_j P_{ij} V_j, \quad P_{ij} = \frac{\exp(S_{ij})}{\sum_k \exp(S_{ik})}$$

The problem: computing $P_{ij}$ requires the full row sum $\sum_k \exp(S_{ik})$, which means we need to see all K before we can compute the output for any Q row.

**The fix**: online softmax. We can compute the output incrementally, processing K in tiles, maintaining a running maximum and normalizer.

## Online softmax derivation

Standard softmax for row $i$, computed in one pass:
$$m_i = \max_j S_{ij}, \quad l_i = \sum_j \exp(S_{ij} - m_i), \quad P_{ij} = \frac{\exp(S_{ij} - m_i)}{l_i}$$

Subtracting $m_i$ is the standard numerical stability trick to prevent overflow.

Now suppose we see the row in two tiles, $[S_{i,0..B}]$ and $[S_{i,B..2B}]$.

After tile 1:
$$m^{(1)} = \max_{j < B} S_{ij}, \quad l^{(1)} = \sum_{j < B} \exp(S_{ij} - m^{(1)})$$

After tile 2, with new max $m^{(2)} = \max(m^{(1)}, \max_{j \geq B} S_{ij})$:
$$l^{(2)} = \underbrace{\exp(m^{(1)} - m^{(2)}) \cdot l^{(1)}}_{\text{rescale old}} + \underbrace{\sum_{j \geq B} \exp(S_{ij} - m^{(2)})}_{\text{new tile}}$$

The $\exp(m^{(1)} - m^{(2)})$ factor rescales the old partial sum to the new reference point.

The same rescaling applies to the accumulated output:
$$O^{(2)} = \frac{\exp(m^{(1)} - m^{(2)}) \cdot O^{(1)} \cdot l^{(1)} + P^{(2)} V^{(2)}}{l^{(2)}}$$

This works for any number of tiles. At the end, $O$ is correct without ever having seen the full row simultaneously.

In [ ]:
import numpy as np

def softmax_reference(x):
    """Standard numerically stable softmax."""
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()

def online_softmax_tiled(x, tile_size):
    """
    Compute softmax(x) tile by tile, maintaining only (m, l) state.
    Returns the normalized weights without ever holding all exp values.
    """
    n = len(x)
    m = -np.inf  # running max
    l = 0.0      # running normalizer
    # We'll accumulate a dummy 'output' (just 1s weighted by softmax) to show correctness
    o = 0.0

    for start in range(0, n, tile_size):
        tile = x[start:start + tile_size]
        m_tile = np.max(tile)
        m_new = max(m, m_tile)

        p_tile = np.exp(tile - m_new)   # unnormalized weights for this tile
        l_new = np.exp(m - m_new) * l + np.sum(p_tile)

        # Rescale accumulated output and add new contribution
        # (Here 'output' = weighted sum of position indices, for illustration)
        v_tile = np.arange(start, start + len(tile), dtype=float)
        o = (np.exp(m - m_new) * o * l + np.dot(p_tile, v_tile)) / l_new

        m, l = m_new, l_new

    return m, l, o

# Verify: tiled output matches reference softmax applied to full array
np.random.seed(42)
x = np.random.randn(16)

weights_ref = softmax_reference(x)
expected_output = np.dot(weights_ref, np.arange(16, dtype=float))

for tile_size in [1, 2, 4, 8, 16]:
    m, l, o = online_softmax_tiled(x, tile_size)
    error = abs(o - expected_output)
    print(f"tile_size={tile_size:2d}  output={o:.6f}  expected={expected_output:.6f}  error={error:.2e}")

## The full tiled attention algorithm

For each Q tile $i$:
1. Initialize: $m \leftarrow -\infty$, $l \leftarrow 0$, $O \leftarrow 0$
2. For each KV tile $j$:
   - $S \leftarrow Q_i K_j^T / \sqrt{d_k}$
   - $m^\text{new} \leftarrow \max(m, \text{rowmax}(S))$
   - $P \leftarrow \exp(S - m^\text{new})$
   - $\alpha \leftarrow \exp(m - m^\text{new})$
   - $l^\text{new} \leftarrow \alpha \cdot l + \text{rowsum}(P)$
   - $O \leftarrow (\alpha \cdot O \cdot l + P V_j) / l^\text{new}$
   - $m, l \leftarrow m^\text{new}, l^\text{new}$
3. Output $O_i$, save $(m_i, l_i)$ for backward recomputation.

VMEM holds: $Q_i$ (one tile), $K_j$ and $V_j$ (one tile each), $O$ accumulator, $(m, l)$ scalars.
Total VMEM: $O(\text{block\_q} \times d_k + \text{block\_kv} \times d_k)$ — independent of sequence length.

In [ ]:
def tiled_attention_numpy(Q, K, V, block_q=4, block_kv=4):
    """
    Flash Attention in plain NumPy, for pedagogical clarity.
    Matches the Pallas kernel algorithm exactly.
    """
    seq_q, d_k = Q.shape
    seq_kv, d_v = V.shape
    scale = 1.0 / np.sqrt(d_k)
    O = np.zeros((seq_q, d_v))

    for q_start in range(0, seq_q, block_q):
        q_end = q_start + block_q
        Q_tile = Q[q_start:q_end]           # (block_q, d_k)

        m = np.full(block_q, -np.inf)       # running max per query row
        l = np.zeros(block_q)              # running normalizer
        O_acc = np.zeros((block_q, d_v))   # running output accumulator

        for kv_start in range(0, seq_kv, block_kv):
            kv_end = kv_start + block_kv
            K_tile = K[kv_start:kv_end]    # (block_kv, d_k)
            V_tile = V[kv_start:kv_end]    # (block_kv, d_v)

            S = Q_tile @ K_tile.T * scale  # (block_q, block_kv)

            m_new = np.maximum(m, S.max(axis=1))     # (block_q,)
            P = np.exp(S - m_new[:, None])            # (block_q, block_kv)
            alpha = np.exp(m - m_new)                 # (block_q,)
            l_new = alpha * l + P.sum(axis=1)         # (block_q,)

            # Rescale accumulated output
            O_acc = (alpha[:, None] * O_acc * l[:, None] + P @ V_tile) / l_new[:, None]

            m, l = m_new, l_new

        O[q_start:q_end] = O_acc

    return O


def reference_attention(Q, K, V):
    """Standard attention for comparison."""
    scale = 1.0 / np.sqrt(Q.shape[1])
    S = Q @ K.T * scale
    P = np.exp(S - S.max(axis=1, keepdims=True))
    P = P / P.sum(axis=1, keepdims=True)
    return P @ V


np.random.seed(0)
SEQ, D = 16, 8
Q = np.random.randn(SEQ, D).astype(np.float32)
K = np.random.randn(SEQ, D).astype(np.float32)
V = np.random.randn(SEQ, D).astype(np.float32)

ref = reference_attention(Q, K, V)

for bq, bkv in [(4, 4), (2, 8), (8, 2), (1, 1), (16, 16)]:
    out = tiled_attention_numpy(Q, K, V, block_q=bq, block_kv=bkv)
    max_err = np.abs(out - ref).max()
    print(f"block_q={bq:2d}  block_kv={bkv:2d}  max_err={max_err:.2e}")

## The recomputation trick for the backward pass

Standard backward pass stores the full $(S, S^2)$ attention matrix from the forward pass, then uses it in the backward. Memory cost: $O(n^2)$ across the forward/backward boundary.

Flash Attention's backward instead:
1. Stores only the log-sum-exp values $m_i$ and $l_i$ (one scalar per query row — $O(n)$ total)
2. During the backward, recomputes $P_{ij} = \exp(S_{ij} - m_i) / l_i$ on the fly

Cost: extra compute in the backward (recompute attention weights from Q, K, V + saved lse).
Benefit: the $O(n^2)$ activation memory is eliminated from the forward/backward boundary.

At long sequence lengths, the memory saving dominates the compute overhead.